# EV Sales SQL Analysis

This notebook verifies eight corrected analytical SQL queries against a local EV sales database dump. The MySQL versions are available in `corrected_queries.sql`; SQLite is used here as a lightweight validation engine.

## Database setup and row-count validation

In [1]:
from pathlib import Path
import re
import sqlite3
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

dump_text = Path("data/ev_sales_db.sql").read_text(encoding="utf-8")
connection = sqlite3.connect(":memory:")

connection.executescript("""
CREATE TABLE dim_date (date TEXT PRIMARY KEY, fiscal_year INTEGER, quarter TEXT);
CREATE TABLE electric_vehicle_sales_by_makers (
    date TEXT, vehicle_category TEXT, maker TEXT, electric_vehicles_sold INTEGER
);
CREATE TABLE electric_vehicle_sales_by_state (
    date TEXT, state TEXT, vehicle_category TEXT,
    electric_vehicles_sold INTEGER, total_vehicles_sold INTEGER
);
""")

for table in ["dim_date", "electric_vehicle_sales_by_makers", "electric_vehicle_sales_by_state"]:
    match = re.search(rf"INSERT INTO `{table}` VALUES (.*?);", dump_text, flags=re.DOTALL)
    if match is None:
        raise ValueError(f"Could not find INSERT statement for {table}")
    connection.executescript(f"INSERT INTO {table} VALUES {match.group(1)};")

def run_query(query):
    return pd.read_sql_query(query, connection)

table_counts = run_query("""
SELECT 'dim_date' AS table_name, COUNT(*) AS rows FROM dim_date
UNION ALL
SELECT 'makers', COUNT(*) FROM electric_vehicle_sales_by_makers
UNION ALL
SELECT 'states', COUNT(*) FROM electric_vehicle_sales_by_state
""")
table_counts

,table_name,rows
0,dim_date,36
1,makers,816
2,states,2445


## Question 1. Unique 2-wheeler makers

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [2]:
query = """SELECT COUNT(DISTINCT maker) AS number_of_makers
FROM electric_vehicle_sales_by_makers
WHERE vehicle_category = '2-Wheelers';"""
run_query(query)

,number_of_makers
0,16


## Question 2. Top three 2-wheeler makers in each fiscal year

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [3]:
query = """WITH maker_sales AS (
 SELECT dd.fiscal_year, evm.maker, SUM(evm.electric_vehicles_sold) AS total_ev_sold
 FROM electric_vehicle_sales_by_makers evm JOIN dim_date dd ON evm.date = dd.date
 WHERE evm.vehicle_category = '2-Wheelers' AND dd.fiscal_year IN (2023, 2024)
 GROUP BY dd.fiscal_year, evm.maker
), ranked AS (
 SELECT *, ROW_NUMBER() OVER (PARTITION BY fiscal_year ORDER BY total_ev_sold DESC) AS maker_rank
 FROM maker_sales
)
SELECT fiscal_year, maker, total_ev_sold FROM ranked
WHERE maker_rank <= 3 ORDER BY fiscal_year, maker_rank;"""
run_query(query)

,fiscal_year,maker,total_ev_sold
0,2023,OLA ELECTRIC,152583
1,2023,OKINAWA,96945
2,2023,HERO ELECTRIC,88993
3,2024,OLA ELECTRIC,322489
4,2024,TVS,180743
5,2024,ATHER,107552


## Question 3. Average monthly total-vehicle sales in FY 2024

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [4]:
query = """WITH monthly_sales AS (
 SELECT evs.date, SUM(evs.total_vehicles_sold) AS total_vehicle_sales
 FROM electric_vehicle_sales_by_state evs JOIN dim_date dd ON evs.date = dd.date
 WHERE dd.fiscal_year = 2024 GROUP BY evs.date
)
SELECT ROUND(AVG(total_vehicle_sales), 0) AS avg_total_sales_per_month FROM monthly_sales;"""
run_query(query)

,avg_total_sales_per_month
0,"1,764,771.00"


## Question 4. Top five states by combined EV penetration in FY 2024

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [5]:
query = """SELECT evs.state,
 ROUND(100.0 * SUM(evs.electric_vehicles_sold) / NULLIF(SUM(evs.total_vehicles_sold), 0), 2) AS penetration_rate
FROM electric_vehicle_sales_by_state evs JOIN dim_date dd ON evs.date = dd.date
WHERE dd.fiscal_year = 2024 AND evs.vehicle_category IN ('2-Wheelers', '4-Wheelers')
GROUP BY evs.state ORDER BY penetration_rate DESC LIMIT 5;"""
run_query(query)

,state,penetration_rate
0,Goa,13.75
1,Kerala,11.59
2,Karnataka,10.18
3,Maharashtra,8.60
4,Delhi,7.71


## Question 5. Highest and lowest state vehicle sales in FY 2023

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [6]:
query = """WITH state_sales AS (
 SELECT evs.state, SUM(evs.total_vehicles_sold) AS total_vehicle_sales
 FROM electric_vehicle_sales_by_state evs JOIN dim_date dd ON evs.date = dd.date
 WHERE dd.fiscal_year = 2023 GROUP BY evs.state
)
SELECT state, total_vehicle_sales,
 CASE WHEN total_vehicle_sales = (SELECT MAX(total_vehicle_sales) FROM state_sales) THEN 'Highest' ELSE 'Lowest' END AS sales_level
FROM state_sales
WHERE total_vehicle_sales = (SELECT MAX(total_vehicle_sales) FROM state_sales)
   OR total_vehicle_sales = (SELECT MIN(total_vehicle_sales) FROM state_sales)
ORDER BY total_vehicle_sales DESC;"""
run_query(query)

,state,total_vehicle_sales,sales_level
0,Uttar Pradesh,2697449,Highest
1,Ladakh,4379,Lowest


## Question 6. Peak and low calendar months across FY 2022 to FY 2024

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [7]:
query = """WITH monthly_ev_sales AS (
 SELECT CAST(strftime('%m', evs.date) AS INTEGER) AS month_number,
        CASE strftime('%m', evs.date)
          WHEN '01' THEN 'January' WHEN '02' THEN 'February' WHEN '03' THEN 'March'
          WHEN '04' THEN 'April' WHEN '05' THEN 'May' WHEN '06' THEN 'June'
          WHEN '07' THEN 'July' WHEN '08' THEN 'August' WHEN '09' THEN 'September'
          WHEN '10' THEN 'October' WHEN '11' THEN 'November' ELSE 'December' END AS month_name,
        SUM(evs.electric_vehicles_sold) AS total_ev_sales
 FROM electric_vehicle_sales_by_state evs JOIN dim_date dd ON evs.date = dd.date
 WHERE dd.fiscal_year BETWEEN 2022 AND 2024
 GROUP BY month_number, month_name
)
SELECT month_name, total_ev_sales,
 CASE WHEN total_ev_sales = (SELECT MAX(total_ev_sales) FROM monthly_ev_sales) THEN 'Peak' ELSE 'Low' END AS season_type
FROM monthly_ev_sales
WHERE total_ev_sales = (SELECT MAX(total_ev_sales) FROM monthly_ev_sales)
   OR total_ev_sales = (SELECT MIN(total_ev_sales) FROM monthly_ev_sales)
ORDER BY total_ev_sales DESC;"""
run_query(query)

,month_name,total_ev_sales,season_type
0,March,291587,Peak
1,June,106709,Low


## Question 7. CAGR for the top four FY 2024 2-wheeler makers

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [8]:
query = """WITH maker_year_sales AS (
 SELECT evm.maker,
  SUM(CASE WHEN dd.fiscal_year = 2022 THEN evm.electric_vehicles_sold ELSE 0 END) AS sales_2022,
  SUM(CASE WHEN dd.fiscal_year = 2024 THEN evm.electric_vehicles_sold ELSE 0 END) AS sales_2024
 FROM electric_vehicle_sales_by_makers evm JOIN dim_date dd ON evm.date = dd.date
 WHERE evm.vehicle_category = '2-Wheelers' AND dd.fiscal_year IN (2022, 2024)
 GROUP BY evm.maker
), top_four AS (
 SELECT * FROM maker_year_sales ORDER BY sales_2024 DESC LIMIT 4
)
SELECT maker, sales_2022, sales_2024,
 ROUND((POWER(sales_2024 * 1.0 / NULLIF(sales_2022, 0), 1.0 / 2) - 1) * 100, 2) AS cagr_percentage
FROM top_four ORDER BY cagr_percentage DESC;"""
run_query(query)

,maker,sales_2022,sales_2024,cagr_percentage
0,OLA ELECTRIC,14401,322489,373.22
1,TVS,9739,180743,330.80
2,BAJAJ,7114,105695,285.45
3,ATHER,19976,107552,132.04


## Question 8. FY 2024 state penetration categories

The query below is the validated SQLite-equivalent calculation. See `corrected_queries.sql` for the corrected MySQL 8 syntax.

In [9]:
query = """WITH state_penetration AS (
 SELECT evs.state, SUM(evs.electric_vehicles_sold) AS total_ev_sales,
  SUM(evs.total_vehicles_sold) AS total_vehicles_sold,
  100.0 * SUM(evs.electric_vehicles_sold) / NULLIF(SUM(evs.total_vehicles_sold), 0) AS penetration_rate
 FROM electric_vehicle_sales_by_state evs JOIN dim_date dd ON evs.date = dd.date
 WHERE dd.fiscal_year = 2024 GROUP BY evs.state
)
SELECT state, total_ev_sales, total_vehicles_sold, ROUND(penetration_rate, 2) AS penetration_rate,
 CASE WHEN penetration_rate > 7 THEN 'Above 7%'
      WHEN penetration_rate > 5 THEN 'Above 5%'
      WHEN penetration_rate > 3 THEN 'Above 3%'
      WHEN penetration_rate > 1 THEN 'Above 1%'
      ELSE 'Below 1%' END AS penetration_category
FROM state_penetration ORDER BY penetration_rate DESC, state;"""
run_query(query)

,state,total_ev_sales,total_vehicles_sold,penetration_rate,penetration_category
0,Goa,10799,78524,13.75,Above 7%
1,Kerala,73938,638114,11.59,Above 7%
2,Karnataka,160989,1581988,10.18,Above 7%
3,Maharashtra,197169,2293994,8.60,Above 7%
4,Delhi,46724,606348,7.71,Above 7%
5,Chandigarh,2877,45147,6.37,Above 5%
6,Odisha,39118,618149,6.33,Above 5%
7,Chhattisgarh,28540,503068,5.67,Above 5%
8,Tamil Nadu,94314,1716940,5.49,Above 5%
9,Puducherry,3098,57692,5.37,Above 5%


## Validation summary

All eight corrected query results were produced directly from the supplied database dump. Question 2 ranks makers separately within each fiscal year. Questions 4 and 8 calculate penetration from aggregated EV and total-vehicle sales, rather than averaging row-level percentages.